# Tiled Softmax Attention

Chapter 10: Matrix Block Partitioning · *Mathematics for AI and Machine Learning* by Fuheng Wu

Online softmax over key/value blocks (the FlashAttention idea) matches full attention exactly.

Companion code for the [interactive demo](https://math4ai.org/companion/chapter-10/tiled-softmax/) on math4ai.org (sign-in required). Run the cell with Shift+Enter; PyTorch is preinstalled on Colab. Change the values at the top and run it again to explore.


In [1]:
import torch

torch.manual_seed(0)
N, d, B = 1024, 64, 128                  # keys, head dim, block size
q = torch.randn(d)
K, V = torch.randn(N, d), torch.randn(N, d)

# Reference: full softmax attention for one query row
s = K @ q / d ** 0.5
exact = torch.softmax(s, dim=0) @ V

# Tiled (online) softmax: one key/value block at a time, never the full row
m = torch.tensor(float("-inf"))
l = torch.tensor(0.0)
o = torch.zeros(d)
for start in range(0, N, B):
    s_blk = K[start:start + B] @ q / d ** 0.5
    m_new = torch.maximum(m, s_blk.max())
    scale = torch.exp(m - m_new)          # rescale what was accumulated so far
    p = torch.exp(s_blk - m_new)
    l = l * scale + p.sum()
    o = o * scale + p @ V[start:start + B]
    m = m_new

tiled = o / l
print("blocks:", N // B, " max |tiled - exact| =", (tiled - exact).abs().max().item())

blocks: 8  max |tiled - exact| = 6.05359673500061e-08
